In [ ]:
## Loading and handeling missing data:
#loading data with pandas
import pandas as pd
DATA_PATH = "TitanicModel/Titanic-Dataset.csv"
df = pd.read_csv(
    DATA_PATH, #example data
    usecols=["Age", "Sex", "Pclass", "Survived"] #selectig coloumns
)
#dropping missing data
df = df.dropna(subset=["Survived"])

#Imputation
from sklearn.impute import SimpleImputer
IMPUTER_STRATEGY = "median" # mean, median, most_frequent, constant
imputer = SimpleImputer(strategy=IMPUTER_STRATEGY)
X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)

In [ ]:
#Inspecting the dataset
print(df.head()) #first 10 rows
print(df.info()) #datatypes
print(df.describe()) # Overview
print(df.isnull().sum()) #how many NaN

In [ ]:
# Defining features and targets
FEATURES = ["Age", "Sex", "Pclass"]
TARGET = "Survived"
X = df[FEATURES].copy()
y = df[TARGET]

# Encoding -> converts categorical values into numerical
X = pd.get_dummies(
    X,
    columns=["Sex"],
    dtype=int
)

In [ ]:
# Splitting the dataset
TEST_SIZE = 0.15  # to evaluate after selecting model
VAL_SIZE = 0.15   # to evalute the model -> hyperparameters
RANDOM_STATE = 42 

from sklearn.model_selection import train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=TEST_SIZE + VAL_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE),
    random_state=RANDOM_STATE,
    stratify=y_temp
)

In [ ]:
# Checking the results
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_pred = model.predict(X_val)

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print(classification_report(y_val, y_pred))

In [ ]:
# Defining a neural network
import torch
import torch.nn as nn

INPUT_SIZE = X_train.shape[1]
HIDDEN_SIZE = 64
OUTPUT_SIZE = 1
DROPOUT = 0.2

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(INPUT_SIZE, HIDDEN_SIZE),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_SIZE, 32),
            nn.ReLU(),
            nn.Linear(32, OUTPUT_SIZE),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

model = NeuralNetwork()

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Hyperparameters
LEARNING_RATE = 0.001
EPOCHS = 500
BATCH_SIZE = 32

# Loss function
criterion = nn.MSELoss()

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Dataset and DataLoader
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(train_dataset)

    # Validation
    model.eval()

    with torch.no_grad():
        val_predictions = model(X_val)
        val_loss = criterion(val_predictions, y_val)

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {avg_loss:.4f} | "
            f"Val Loss: {val_loss.item():.4f}"
        )